In [ ]:
import os
import re
import gc
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path

In [ ]:
def weighted_global_efficiency(G, weight='distance'):
    n = len(G)
    if n < 2: return 0.0
    total = 0.0
    for node in G.nodes():
        lengths = nx.single_source_dijkstra_path_length(G, node, weight=weight)
        total += sum(1.0 / d for target, d in lengths.items() if target != node and d > 0)
    return total / (n * (n - 1))

def compute_features(mat, ratio_thres):
    G = nx.from_numpy_array(mat)
    for u, v, data in G.edges(data=True):
        data['distance'] = 1 / max(data['weight'], EPSILON)
    cluster_coeff = nx.clustering(G, weight='weight')
    strengths = dict(G.degree(weight='weight'))
    try:
        eigen_central = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)
    except Exception:
        eigen_central = {node: 0.0 for node in G.nodes()}
    path_length = np.nan if not nx.is_connected(G) else nx.average_shortest_path_length(G, weight='distance')

    thres = ratio_thres * mat.max()
    mat_ = np.where(mat > thres, mat, 0)
    G_ = nx.from_numpy_array(mat_)
    for u, v, data in G_.edges(data=True):
        data['distance'] = 1 / max(data['weight'], EPSILON)
    between_central = nx.betweenness_centrality(G_, weight='distance', normalized=True)
    dens = nx.density(G_)
    wge = weighted_global_efficiency(G_, weight='distance')

    node_features = {
        'Clustering_Coeff': cluster_coeff,
        'Strength': strengths,
        'Eigen_Centrality': eigen_central,
        'Betweenness_Centrality': between_central
    }
    graph_features = {
        'Density': dens,
        'Weighted_Global_Efficiency': wge,
        'Characteristic_Path_Length': path_length
    }
    return node_features, graph_features

def compute_all_features(pli_current, ratio_thres):
    node_features, graph_features = compute_features(pli_current, ratio_thres)
    flat_dict = {}
    for feat_name, values in node_features.items():
        for node_id, value in values.items():
            flat_dict[f"Node_{node_id}_{feat_name}"] = value
    flat_dict.update(graph_features)
    return flat_dict

In [ ]:
# ==========================================================================
# FEATURE EXTRACTION
# ==========================================================================
def extract_graph_features(folder_path,step=None,ratio_thres=0.5):
    
    folder = Path(folder_path)
    files = sorted([f for f in folder.glob('*.npy')])
    
    if not files:
        return pd.DataFrame()

    rows_list = []
    for file_path in files:
        
        band = file_path.name.split('.')[0]            # 'alpha'
        print('Band: ', band)
        sz = file_path.parent.parent.name
        matrices = np.load(file_path)
                
        for epoch_idx, matrix in enumerate(matrices):
            feat_dict = compute_all_features(matrix, ratio_thres)
            row_data = {
                    'SeizureID': sz,
                    'Band': band,
                    'Epoch': epoch_idx+step
                }
            print('Epoch', epoch_idx+step)
            row_data.update(feat_dict)
            rows_list.append(row_data)

    final_df = pd.DataFrame(rows_list)
    return final_df, matrices.shape[0]
# ==========================================================================
# MAIN EXECUTION FOR TESTING DATA (FOR FAR/H CALCULATION)
# ==========================================================================
if __name__ == '__main__':
    for mode in ['wPLI', 'PLI']:
        base_path = Path(f'/kaggle/input/datasets/ngoclannguyen00/chb23-connectivity/CHB23_connectivity_Testing/{mode}_matrix')
        output_dir = Path(f'{mode}_Testing_Features_Nodal_Thres_0.6_chb23')
    
        for chb_folder in sorted(base_path.iterdir()):
            if not chb_folder.is_dir():
                continue
                
            num_chb = chb_folder.name  # 'chb01', 'chb02'...
            chb_output = output_dir
            chb_output.mkdir(parents=True, exist_ok=True)
            print(f'Processing {num_chb}...')
            
            df_list = []
            for sz_path in chb_folder.glob('*'):
                
                sz = sz_path.name
                print('Seizure: ', sz)
                prev_batch_size = 0
                step=0
                for batch_path in sorted(sz_path.glob('*')):
                    step += prev_batch_size
                    if batch_path.is_dir():
                        features, prev_batch_size = extract_graph_features(batch_path, step=step, ratio_thres=0.6)
                        if not features.empty:
                            df_list.append(features)
                    
            if df_list:
                df_result = pd.concat(df_list, ignore_index=True)
                save_path = chb_output / f'{num_chb}.csv'
                df_result.to_csv(save_path, index=False)
                print(f'  [OK] Completed {num_chb} -> Saved to {save_path}')
                    
                del df_list, df_result
            gc.collect()

In [ ]:
# =====================================================
# 1. FEATURES EXTRACTION
# =====================================================
def extract_graph_features(folder_path,label, step=None,ratio_thres=0.7):
    
    folder = Path(folder_path)
    files = sorted([f for f in folder.glob('*.npy')])
    
    if not files:
        return pd.DataFrame()

    rows_list = []
    for file_path in files:
        band = file_path.name.split('.')[0]            # 'alpha'
        print('Band: ', band)
        file_id = file_path.parent.name # 'seizure ID or batch index'
        
        matrices = np.load(file_path)
                
        for epoch_idx, matrix in enumerate(matrices):
            feat_dict = compute_all_features(matrix, ratio_thres)
            if label=='preictal':
                row_data = {
                    'SeizureID': file_id,
                    'Band': band,
                    'Epoch': epoch_idx
                }
            else:
                row_data = {
                    'Band': band,
                    'Epoch': epoch_idx+step
                }
                print('Epoch', epoch_idx+step)
            row_data.update(feat_dict)
            rows_list.append(row_data)

    final_df = pd.DataFrame(rows_list)
    return final_df, matrices.shape[0]

# ==========================================================================
# MAIN EXECUTION (FOR TRAINING DATA)
# ==========================================================================
if __name__ == '__main__':
    for mode in ['PLI', 'wPLI']:
        base_path = Path(f'/kaggle/input/datasets/ngoclannguyen00/chb23-connectivity/CHB23_connectivity_training/{mode}_matrix')
        output_dir = Path(f'{mode}_Training_Features_Nodal_Thres_0.6_chb23')
        output_dir.mkdir(parents=True, exist_ok=True)
        for chb_folder in sorted(base_path.iterdir()):
            if not chb_folder.is_dir():
                continue
                
            num_chb = chb_folder.name  # 'chb01', 'chb02'...
            chb_output = output_dir / f'{num_chb}'
            chb_output.mkdir(parents=True, exist_ok=True)
            print(f'Processing {num_chb}...')
            # Duyệt qua tất cả các thư mục band (chb01/seizure_XX/band_name)
            # glob('*/*') sẽ khớp với: seizure_01/alpha, seizure_01/beta, ...
            for label_path in chb_folder.glob('*'):
                df_list = []
                label = label_path.name
                print('Label: ', label)
                prev_batch_size = 0
                step=0
                for epochs_path in sorted(label_path.glob('*')):
                    if label=='preictal':
                        print('Seizure ID: ', epochs_path.name)
                    batch_idx = epochs_path.name
                    if label=='interictal':
                        step += prev_batch_size
                    else:
                        step==None
                    if epochs_path.is_dir():
                        features, prev_batch_size = extract_graph_features(epochs_path, label, step=step, ratio_thres=0.6)
                        if not features.empty:
                            df_list.append(features)
                    
                if df_list:
                    df_result = pd.concat(df_list, ignore_index=True)
                    save_path = chb_output / f'{label}.csv'
                    df_result.to_csv(save_path, index=False)
                    print(f'  [OK] Completed {num_chb} -> Saved to {save_path}')
                    
                    del df_list, df_result
                gc.collect()